<a href="https://colab.research.google.com/github/Shrideshi1/multi-label-email-risk-detection/blob/main/notebooks/05_2_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Load Saved Outputs

!pip install -q transformers

import os
import re
import joblib
import torch
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import hstack, csr_matrix
from sklearn.metrics import log_loss
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    classification_report,
    multilabel_confusion_matrix
)

drive.mount("/content/drive")

PROJECT_DIR = (
    "/content/drive/.shortcut-targets-by-id/"
    "1SLhiH7VulyiPZ7L846kJBEbN1pa4c4zU/"
    "Multi_Label_Email_Risk_Detection"
)

NOTEBOOKS_DIR = f"{PROJECT_DIR}/notebooks"
DATA_PROCESSED_DIR = f"{PROJECT_DIR}/data/processed"
FEATURE_DIR = f"{DATA_PROCESSED_DIR}/features"
MODELS_DIR = f"{PROJECT_DIR}/models"
REPORTS_DIR = f"{PROJECT_DIR}/reports"
FIGURES_DIR = f"{PROJECT_DIR}/figures"

os.chdir(PROJECT_DIR)
os.makedirs(REPORTS_DIR, exist_ok=True)

risk_cols = [
    "financial_risk",
    "credential_risk",
    "customer_info_risk",
    "proprietary_risk",
    "legal_risk",
    "attachment_risk",
    "phishing_spam_risk"
]

X_test = joblib.load(f"{FEATURE_DIR}/X_test_features.pkl")
Y_test = np.asarray(joblib.load(f"{FEATURE_DIR}/y_test.pkl")).astype(int)
tfidf_vectorizer = joblib.load(f"{FEATURE_DIR}/tfidf_vectorizer.pkl")
metadata_scaler = joblib.load(f"{FEATURE_DIR}/metadata_scaler.pkl")
metadata_df = pd.read_csv(f"{FEATURE_DIR}/metadata_features.csv")
rule_features_df = pd.read_csv(f"{FEATURE_DIR}/rule_features.csv")

models = {
    "Logistic Regression": joblib.load(f"{MODELS_DIR}/logistic_regression_model.pkl"),
    "Random Forest": joblib.load(f"{MODELS_DIR}/random_forest_model.pkl"),
    "SVM": joblib.load(f"{MODELS_DIR}/svm_model.pkl"),
    "XGBoost": joblib.load(f"{MODELS_DIR}/xgboost_model.pkl"),
    "Neural Network": tf.keras.models.load_model(
        f"{MODELS_DIR}/multi_label_email_risk_mlp.keras"
    )
}

prediction_files = {
    "DistilBERT": "distilbert_predictions.pkl",
    "Random Forest": "random_forest_predictions.pkl",
    "Neural Network": "mlp_predictions.pkl",
    "Logistic Regression": "logistic_regression_predictions.pkl",
    "XGBoost": "xgboost_predictions.pkl",
    "SVM": "svm_predictions.pkl"
}

predictions = {
    model_name: np.asarray(joblib.load(f"{REPORTS_DIR}/{filename}"))
    for model_name, filename in prediction_files.items()
}

distilbert_path = f"{MODELS_DIR}/distilbert_email_risk"
distilbert_tokenizer = AutoTokenizer.from_pretrained(distilbert_path)
distilbert_model = AutoModelForSequenceClassification.from_pretrained(distilbert_path)
distilbert_model.eval()

print("Current project folder:", os.getcwd())
print("Notebook files:", os.listdir(NOTEBOOKS_DIR))
print("Feature files:", os.listdir(FEATURE_DIR))
print("Model files:", os.listdir(MODELS_DIR))
print("Report files:", os.listdir(REPORTS_DIR))
print("Test feature shape:", X_test.shape)
print("Test label shape:", Y_test.shape)
print("Loaded models:", list(models))
print("Loaded predictions:", list(predictions))

Mounted at /content/drive


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Current project folder: /content/drive/.shortcut-targets-by-id/1SLhiH7VulyiPZ7L846kJBEbN1pa4c4zU/Multi_Label_Email_Risk_Detection
Notebook files: ['01_data_preparation.ipynb', '02_feature_engineering.ipynb', '03.2_model_training_CV.ipynb', '04_evaluation.ipynb', '03_model_training.ipynb', '05_demo_test', '05.2_Demo.ipynb']
Feature files: ['X_features.pkl', 'metadata_features.csv', 'y_labels.pkl', 'rule_features.csv', 'X_train_features.pkl', 'X_test_features.pkl', 'metadata_scaler.pkl', 'tfidf_vectorizer.pkl', 'y_test.pkl', 'y_train.pkl']
Model files: ['multi_label_email_risk_mlp.keras', 'distilbert_email_risk', 'svm_model (1).pkl', 'xgboost_model (1).pkl', 'logistic_regression_model_og.pkl', 'mlp_model_og.pkl', 'svm_model_og.pkl', 'xgboost_model_og.pkl', 'random_forest_model.pkl', 'random_forest_model_og.pkl', 'logistic_regression_model.pkl', 'mlp_model.pkl', 'svm_model.pkl', 'xgboost_model.pkl']
Report files: ['nn_overall_metrics.csv', 'nn_per_class_metrics.csv', 'nn_binary_cross_entr

In [ ]:
# 2. Enter New Email

import ipywidgets as widgets
from IPython.display import display

email_box = widgets.Textarea(
    value=
    """Subject: URGENT: Your Account will be Deactivated in 24 Hours
Body:
Dear User,
We noticed some unusual activity on your account. To protect your privacy, we have temporarily locked your access.
Please click the link below to verify your identity and restore your account. Failure to do this within 24 hours will result in permanent account deletion.
Verify Your Account Now
Sincerely,
Customer Support Team""",
    placeholder="Paste a new email here...",
    description="Email:",
    layout=widgets.Layout(width="100%", height="300px")
)

display(email_box)

Textarea(value='Subject: URGENT: Your Account will be Deactivated in 24 Hours\nBody:\nDear User,\nWe noticed s…

In [ ]:
## 3. Prepare New Email Features

email_text = email_box.value.strip()

if not email_text:
    raise ValueError("Paste an email into the text box before running this block.")

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " URL ", text)
    text = re.sub(r"\S+@\S+", " EMAIL ", text)
    text = re.sub(r"[_\-=\*]{3,}", " ", text)
    text = re.sub(r"\b_+\w+_*\b", " ", text)
    text = re.sub(r"\b\w+_+\w+\b", " ", text)
    text = re.sub(r"\d+", " NUMBER ", text)
    text = re.sub(r"[^a-zA-Z0-9\s\.\-$]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def create_metadata_features(df):

    return pd.DataFrame({
        "text_length": df["text"].astype(str).str.len(),
        "word_count": df["text"].astype(str).str.split().str.len(),
        "num_digits": df["text"].astype(str).str.count(r"\d"),
        "num_dollar_signs": df["text"].astype(str).str.count(r"\$"),
        "num_uppercase": df["text"].astype(str).str.count(r"[A-Z]"),
        "num_exclamation": df["text"].astype(str).str.count(r"!"),
        "num_question": df["text"].astype(str).str.count(r"\?")
    })

def create_rule_features(df):
    text_lower = df["text"].astype(str).str.lower()

    return pd.DataFrame({
        "has_attachment_ext": text_lower.str.contains(r"\.pdf|\.docx|\.xlsx|\.csv|\.zip|\.pptx", regex=True).astype(int),
        "has_money": text_lower.str.contains(r"\$| revenue | budget | valuation | invoice | payment ", regex=True).astype(int),
        "has_credential_terms": text_lower.str.contains(r"password|token|api key|secret|credential|oauth|vpn|encryption key|root", regex=True).astype(int),
        "has_customer_terms": text_lower.str.contains(r"customer|client|account|employee|vendor|partner|payroll", regex=True).astype(int),
        "has_legal_terms": text_lower.str.contains(r"contract|nda|liability|clause|agreement|compliance|legal|indemnification", regex=True).astype(int),
        "has_internal_terms": text_lower.str.contains(r"internal|confidential|do not forward|do not distribute|not for circulation", regex=True).astype(int)
    })

metadata_columns = ["text_length", "word_count", "num_digits", "num_dollar_signs", "num_uppercase", "num_exclamation", "num_question"]
rule_columns = ["has_attachment_ext", "has_money", "has_credential_terms", "has_customer_terms", "has_legal_terms", "has_internal_terms"]

email_df = pd.DataFrame({"text": [email_text]})
email_df["clean_text"] = email_df["text"].apply(clean_text)

email_metadata_df = create_metadata_features(email_df)[metadata_columns]
email_rule_df = create_rule_features(email_df)[rule_columns]

email_tfidf = tfidf_vectorizer.transform(email_df["clean_text"])
email_metadata = csr_matrix(metadata_scaler.transform(email_metadata_df))
email_rules = csr_matrix(email_rule_df.astype(float).values)
email_features = hstack([email_tfidf, email_metadata, email_rules], format="csr")

if email_features.shape[1] != X_test.shape[1]:
    raise ValueError(f"Feature mismatch: new email has {email_features.shape[1]} features but models expect {X_test.shape[1]}.")

print("Current email preview:")
print(email_text[:500])
print("\nCleaned text:")
print(email_df["clean_text"].iloc[0][:500])
print("\nRule indicators:")
display(email_rule_df.T.rename(columns={0: "Value"}))
print("Feature shape:", email_features.shape)
print("Nonzero features:", email_features.nnz)

Current email preview:
Subject: Project Atlas Design Package – Internal
Review​

Engineering Team,​

The latest Project Atlas system architecture,
component specifications, manufacturing
tolerances, and implementation roadmap have been
finalized for internal review.​

These materials contain proprietary design
information, unreleased product details, supplier
integration requirements, and confidential
technical specifications.​

Do not forward, copy, or distribute this information
outside the Project Atlas engineering

Cleaned text:
subject project atlas design package internal review engineering team the latest project atlas system architecture component specifications manufacturing tolerances and implementation roadmap have been finalized for internal review. these materials contain proprietary design information unreleased product details supplier integration requirements and confidential technical specifications. do not forward copy or distribute this information outside the projec

,Value
has_attachment_ext,1
has_money,0
has_credential_terms,0
has_customer_terms,0
has_legal_terms,0
has_internal_terms,1


Feature shape: (1, 10013)
Nonzero features: 48


In [ ]:
# 4. Generate Fresh Model Probabilities

def get_multioutput_probabilities(model, features):
    return np.array([
        estimator.predict_proba(features)[0, list(estimator.classes_).index(1)]
        if 1 in estimator.classes_
        else float(estimator.classes_[0])
        for estimator in model.estimators_
    ])

print("Current email preview:")
print(email_text[:300])
print("Feature shape:", email_features.shape)
print("Nonzero features:", email_features.nnz)

model_probabilities = {}

model_probabilities["Logistic Regression"] = get_multioutput_probabilities(
    models["Logistic Regression"],
    email_features
)

model_probabilities["Random Forest"] = get_multioutput_probabilities(
    models["Random Forest"],
    email_features
)

model_probabilities["XGBoost"] = get_multioutput_probabilities(
    models["XGBoost"],
    email_features
)

model_probabilities["Neural Network"] = models[
    "Neural Network"
].predict(
    email_features.toarray(),
    verbose=0
)[0]

distilbert_inputs = distilbert_tokenizer(
    email_text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)

with torch.no_grad():
    distilbert_logits = distilbert_model(
        **distilbert_inputs
    ).logits

model_probabilities["DistilBERT"] = torch.sigmoid(
    distilbert_logits
).cpu().numpy()[0]

probability_df = pd.DataFrame(
    model_probabilities,
    index=risk_cols
)

display(probability_df.round(4))

Current email preview:
Subject: Project Atlas Design Package – Internal
Review​

Engineering Team,​

The latest Project Atlas system architecture,
component specifications, manufacturing
tolerances, and implementation roadmap have been
finalized for internal review.​

These materials contain proprietary design
information
Feature shape: (1, 10013)
Nonzero features: 48


,Logistic Regression,Random Forest,XGBoost,Neural Network,DistilBERT
financial_risk,0.0000,0.090,0.0011,0.0000,0.0196
credential_risk,0.0001,0.055,0.0000,0.0000,0.0081
customer_info_risk,0.0000,0.000,0.0000,0.0000,0.0044
proprietary_risk,0.8303,0.275,0.0475,0.9999,0.0195
legal_risk,0.0000,0.110,0.0055,0.0000,0.0208
attachment_risk,0.9694,0.175,0.0073,0.0000,0.0255
phishing_spam_risk,0.0000,0.065,0.0880,0.0000,0.0016


In [ ]:
risk_names = {
    "financial_risk": "Financial Information",
    "credential_risk": "Credentials or Authentication",
    "customer_info_risk": "Customer or Employee Information",
    "proprietary_risk": "Classified or Proprietary Information",
    "legal_risk": "Legal or Contract Information",
    "attachment_risk": "Sensitive Attachment",
    "phishing_spam_risk": "Phishing or Spam"
}

available_probability_models = list(model_probabilities)

probability_df["Ensemble Average"] = probability_df[available_probability_models].mean(axis=1)
probability_df["Final Probability"] = probability_df[available_probability_models].max(axis=1)
probability_df["Triggering Model"] = probability_df[available_probability_models].idxmax(axis=1)

results_df = probability_df.reset_index().rename(columns={"index": "Risk Code"})
results_df["Risk Category"] = results_df["Risk Code"].map(risk_names)
results_df["Probability Value"] = results_df["Final Probability"].clip(0, 1)
results_df["Threshold"] = 0.50
results_df["Prediction"] = np.where(results_df["Probability Value"] >= results_df["Threshold"], "Detected", "Not Detected")
results_df["Probability"] = results_df["Probability Value"].map("{:.2%}".format)
results_df["Threshold"] = results_df["Threshold"].map("{:.0%}".format)
results_df["Ensemble Average"] = results_df["Ensemble Average"].map("{:.2%}".format)
results_df = results_df.sort_values("Probability Value", ascending=False)

spam_probability = probability_df.loc["phishing_spam_risk", "Final Probability"]
proprietary_probability = probability_df.loc["proprietary_risk", "Final Probability"]

confidential_risks = ["financial_risk", "credential_risk", "customer_info_risk", "proprietary_risk", "legal_risk", "attachment_risk"]
highest_leak_category = probability_df.loc[confidential_risks, "Final Probability"].idxmax()
highest_leak_probability = probability_df.loc[highest_leak_category, "Final Probability"]

print(f"Phishing or Spam Probability: {spam_probability:.2%} ({probability_df.loc['phishing_spam_risk', 'Triggering Model']})")
print(f"Classified or Proprietary Probability: {proprietary_probability:.2%} ({probability_df.loc['proprietary_risk', 'Triggering Model']})")
print(f"Highest Information-Leak Risk: {risk_names[highest_leak_category]} ({highest_leak_probability:.2%}, {probability_df.loc[highest_leak_category, 'Triggering Model']})")

display(results_df[["Risk Category", "Probability", "Threshold", "Triggering Model", "Ensemble Average", "Prediction"]])

Phishing or Spam Probability: 8.80% (XGBoost)
Classified or Proprietary Probability: 99.99% (Neural Network)
Highest Information-Leak Risk: Classified or Proprietary Information (99.99%, Neural Network)


,Risk Category,Probability,Threshold,Triggering Model,Ensemble Average,Prediction
3,Classified or Proprietary Information,99.99%,50%,Neural Network,43.44%,Detected
5,Sensitive Attachment,96.94%,50%,Logistic Regression,23.54%,Detected
4,Legal or Contract Information,11.00%,50%,Random Forest,2.73%,Not Detected
0,Financial Information,9.00%,50%,Random Forest,2.21%,Not Detected
6,Phishing or Spam,8.80%,50%,XGBoost,3.09%,Not Detected
1,Credentials or Authentication,5.50%,50%,Random Forest,1.26%,Not Detected
2,Customer or Employee Information,0.44%,50%,DistilBERT,0.09%,Not Detected
